# FinSight — teaching Qwen 2.5 3B to read contracts  `[B]`

Phase 10. Runs on a **free Colab or Kaggle T4**. About an hour end to end,
most of it waiting.

## What this produces

Not a new model — a **patch** (a LoRA adapter) of roughly 50–100 MB that sits on
top of the public Qwen weights and adjusts how it reads contracts. It uploads to
your HuggingFace account, and `LLM_MODEL` is the only thing that changes in the
app afterwards.

## The one rule

**`eval_set.jsonl` must never be trained on.** It is the exam: 22 real contracts,
human-approved, sealed before any training example existed. If it leaks into
training, the Phase 11 comparison stops meaning anything — and it fails in the
worst direction, scoring *high*, so nothing warns you.

Cell 5 checks for that leak and **stops the notebook** if it finds one. Do not
skip or "fix" it.

## Before you start

1. **Runtime → Change runtime type → T4 GPU → Save.** (Kaggle: Accelerator → GPU T4, Internet **on**.)
2. Add `HF_TOKEN` to the 🔑 **Secrets** panel (left sidebar) and switch
   **Notebook access** on. It needs **write** permission.
3. Run the cells in order. Leave the tab open while training.

If the tab drops mid-run, restart and re-run — checkpoints are saved every epoch.

## 1 · Check we actually have a GPU

In [ ]:
import subprocess, sys

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit("No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.")
print(out.stdout.split("\n")[8] if len(out.stdout.split("\n")) > 8 else out.stdout)

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
print("memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2 · Install Unsloth

Unsloth makes the training fit on a free T4. This is the slow cell — 3–5 minutes.

> **Known trap (issue #50):** installing a new torch leaves Colab's preinstalled
> `torchaudio` a version behind, and `transformers` then refuses to import.
> We remove it — this project serves text and never touches audio.

In [ ]:
%%capture
!pip install -q unsloth
!pip uninstall -y -q torchaudio || true

In [ ]:
# Re-check after the install: a failed install here is silent until much later.
import importlib, sys
for mod in ("unsloth", "transformers", "trl", "peft", "datasets"):
    try:
        m = importlib.import_module(mod)
        print(f"  ok   {mod:14} {getattr(m, '__version__', '?')}")
    except Exception as exc:
        print(f"  FAIL {mod:14} {exc}")
        sys.exit("Install did not complete. Re-run the cell above.")

## 3 · Get the data across

**This notebook is not running on your laptop.** It is a computer in a Google
data centre, and it cannot see your files. They have to be sent to it.

Two files travel by hand. Run the cell below and it will offer you a
**Choose Files** button. Pick these two from your machine:

```
training/data/train.jsonl        (73 examples to learn from)
training/data/val.jsonl          (12 to check progress against)
```

Hold Ctrl (or Cmd) to select both at once. They upload in a few seconds —
together they are about 166 KB.

**The exam paper comes on its own.** `eval_set.jsonl` is committed to the
project, so the next cell clones the repo and it arrives with no help from you.
Same for `core/ai/prompts.py`, which this notebook needs so it trains on the
*same* instructions the live app sends. If you would rather upload the exam by
hand anyway, the button accepts it — nothing breaks.

> **They disappear when the session ends.** Colab wipes the machine when you
> close it, so if you come back tomorrow you upload again. That is normal, not
> something broken.

In [ ]:
import os
from pathlib import Path

BY_HAND = ["train.jsonl", "val.jsonl"]        # generated, deliberately not in git
FROM_GIT = ["eval_set.jsonl"]                 # committed; the next cell fetches it
WANTED = BY_HAND + FROM_GIT
DATA = Path("data"); DATA.mkdir(exist_ok=True)

def find(name):
    # Look everywhere a file could plausibly have landed: the data folder, the
    # working directory (where drag-and-drop puts things), /content, the repo.
    for candidate in (DATA / name, Path(name), Path("/content") / name,
                      Path("Fin/training/data") / name):
        if candidate.exists():
            return candidate
    return None

missing = [n for n in BY_HAND if find(n) is None]
if missing:
    print("missing:", ", ".join(missing))
    try:
        from google.colab import files
        print("\nPick them with the button below (Ctrl/Cmd-click for both at once):")
        uploaded = files.upload()
        print("\nreceived:", list(uploaded))
    except ImportError:
        print("Not on Colab — put the files beside this notebook, or let the next cell fetch them.")
else:
    print("training data already here:", {n: str(find(n)) for n in BY_HAND})

# Normalise: whatever arrived, wherever it landed, ends up in data/
for name in WANTED:
    found = find(name)
    if found and found.resolve() != (DATA / name).resolve():
        (DATA / name).write_bytes(found.read_bytes())
        print(f"  {name} -> data/{name}")

### What the repo supplies

This cell clones the project and takes three things out of it:

1. **`eval_set.jsonl`** — the exam, committed on purpose so it always arrives.
2. **`core/ai/prompts.py`** — the *live app's* instructions to the model. The
   model must be taught with the same instructions it will be tested and used
   with, or the training is aimed at a prompt that never happens.
3. **`train.jsonl` / `val.jsonl`** — only if you uploaded nothing. It tries your
   HuggingFace account first, then regenerates them from `build_pairs.py`
   (deterministic, no API key needed, but without the DeepSeek wording variety).

If the upload above worked, most of this cell finds things already in place and
does nothing.

In [ ]:
import importlib.util, json, os, sys
from pathlib import Path

HF_DATASET = "ibrahim404/finsight-contract-pairs"   # set to None to skip
DATA = Path("data"); DATA.mkdir(exist_ok=True)

def have(*names):
    return all((DATA / n).exists() for n in names)

def ensure_repo():
    """Clone once. Everything the repo supplies goes through here."""
    if not Path("Fin").exists():
        !git clone -q https://github.com/maybethemuhammadibrahim/Fin.git 2>/dev/null || true
    return Path("Fin").exists()

# ---- try the Hub ----
if HF_DATASET and not have("train.jsonl", "val.jsonl"):
    try:
        from google.colab import userdata
        os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
    except Exception:
        pass
    try:
        from huggingface_hub import snapshot_download
        got = snapshot_download(HF_DATASET, repo_type="dataset",
                                token=os.environ.get("HF_TOKEN"), local_dir=str(DATA))
        print("pulled from the Hub:", got)
    except Exception as exc:
        print("Hub not available:", str(exc)[:120])

# ---- fall back to the repo, regenerating the pairs ----
if not have("train.jsonl", "val.jsonl"):
    ensure_repo()
    if (Path("Fin/training/data/train.jsonl")).exists():
        for n in ("train.jsonl", "val.jsonl"):
            (DATA / n).write_bytes(Path(f"Fin/training/data/{n}").read_bytes())
        print("copied from the repo")
    else:
        print("regenerating the pairs (deterministic, no API needed)")
        !cd Fin && python training/build_pairs.py --count 85
        for n in ("train.jsonl", "val.jsonl"):
            (DATA / n).write_bytes(Path(f"Fin/training/data/{n}").read_bytes())

# ---- the exam set, fetched INDEPENDENTLY of the training data ----
# This used to hang off the training-data branch above, so uploading train/val
# by hand meant the repo was never cloned, eval_set.jsonl never arrived, and
# the leak check in cell 5 passed with nothing to check. A safety check that
# cannot fail is worse than no safety check.
if not (DATA / "eval_set.jsonl").exists():
    ensure_repo()
    src = Path("Fin/training/data/eval_set.jsonl")
    if src.exists():
        (DATA / "eval_set.jsonl").write_bytes(src.read_bytes())
        print("exam set: from the repo")
    else:
        print("exam set: NOT FOUND — upload training/data/eval_set.jsonl into the file panel")

# ---- the live app's prompt, so training matches serving ----
# core/ai/prompts.py imports nothing but __future__, so loading it straight off
# disk costs no dependencies and skips the package __init__ chain entirely.
# We refuse to train without it: a run with the wrong prompt does not fail, it
# succeeds at the wrong thing, and nothing downstream would ever tell you.
ensure_repo()
prompt_file = Path("Fin/core/ai/prompts.py")
if not prompt_file.exists():
    raise SystemExit(
        "STOP: core/ai/prompts.py did not arrive.\n"
        "Training with a different prompt from the one the app sends would produce\n"
        "an adapter tuned for a prompt that never occurs, and the Phase 11 comparison\n"
        "would no longer be one variable. Upload core/ai/prompts.py to Fin/core/ai/,\n"
        "or fix the clone, then re-run this cell."
    )
_spec = importlib.util.spec_from_file_location("finsight_prompts", prompt_file)
prompts = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(prompts)
print(f"runtime prompt: {prompts.PROMPT_VERSION} "
      f"({len(prompts.EXTRACTION_SYSTEM):,} chars of system instructions)")

def load(name):
    path = DATA / name
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()] if path.exists() else []

train_rows, val_rows, eval_rows = load("train.jsonl"), load("val.jsonl"), load("eval_set.jsonl")
print(f"\ntrain {len(train_rows)}  val {len(val_rows)}  eval(held out) {len(eval_rows)}")
assert train_rows and val_rows, "No training data. Upload train.jsonl and val.jsonl into the file panel."

## 4 · Look at one example

Worth ten seconds. If the input is not a contract or the output is not the
matching JSON, stop — everything downstream inherits this.

In [ ]:
row = train_rows[0]
print("INPUT (first 500 chars):\n", row["input"][:500], "\n")
print("EXPECTED OUTPUT:\n", json.dumps(json.loads(row["output"]), indent=2)[:600])

# row["instruction"] exists but is NOT used for training — section 7 replaces it
# with the app's own extraction_user(), so the model is taught on the real thing.
print("\n(unused) instruction field:", row["instruction"])

## 5 · The leak check — **do not skip this**

Every training example is compared against every exam question. If any exam text
appears in training, this cell stops the notebook.

A leak does not announce itself. It just produces a wonderful score.

In [ ]:
import re, sys

# A leak check with no exam questions is not a pass — it is a check that cannot
# fail. Refuse to continue rather than print a reassuring line about zero of zero.
if not eval_rows:
    sys.exit(
        "STOP: eval_set.jsonl is missing, so there is nothing to check for leaks.\n"
        "Upload training/data/eval_set.jsonl into the file panel on the left and\n"
        "re-run the data cell. Do NOT skip this and train anyway."
    )

def fold(s):
    return re.sub(r"\s+", " ", s).strip().lower()

train_text = [fold(r["input"]) for r in train_rows + val_rows]
leaks = []
for row in eval_rows:
    body = fold(row["input"])
    probes = [body[i:i + 80] for i in range(0, max(1, len(body) - 80), 400)][:12]
    for probe in probes:
        if not probe.strip():
            continue
        if any(probe in t for t in train_text):
            leaks.append(row.get("source", "?"))
            break

if leaks:
    print("LEAK — these exam contracts appear in the training data:")
    for name in leaks:
        print("  ", name)
    sys.exit("STOP. Training now would make the Phase 11 result meaningless.")
print(f"clean: none of the {len(eval_rows)} exam contracts appear in the {len(train_text)} training examples")

## 6 · Load Qwen 2.5 3B

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL   = "unsloth/Qwen2.5-3B-Instruct"   # 4-bit ready; same weights as Qwen/Qwen2.5-3B-Instruct
LORA_RANK    = 16

# 4096, not 2048. Measured with the real Qwen tokenizer against the real files,
# using the app's own v4 prompt (1,343 tokens of system instructions on their own):
#   training examples   max 1,973 tokens  -> 2048 would fit, with 75 to spare
#   exam prompts        max 2,378 tokens  -> 6 of the 22 do NOT fit in 2048
# The exam is never trained on, but section 9 runs one through the model, and
# unsloth caps the context at this value for the whole session. 2048 would have
# silently truncated the longest exam contracts — losing the clause and blaming
# the model for missing it. Padding is per batch, so the extra headroom is
# almost free in memory.
MAX_SEQ_LEN  = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
    dtype          = None,   # let it pick: T4 has no bfloat16
)

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    lora_alpha = LORA_RANK,
    lora_dropout = 0.0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",   # what makes this fit on a T4
    random_state = 20260817,
)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7 · Shape the examples

Each pair becomes a two-turn conversation: the contract in, the JSON out.

**The instructions come from `core/ai/prompts.py`, not from this notebook.**
That is the whole point. `EXTRACTION_SYSTEM` (v4, ~5,600 characters of numbered
rules, a JSON skeleton and a worked example) and `extraction_user()` are exactly
what `core/ai/contract_extractor.py` sends every time the app reads a contract,
and exactly what `scripts/eval_extraction.py` will send at Phase 11.

Writing a shorter prompt here would teach the model to answer a question nobody
asks it — and Phase 11 would then be comparing two things at once instead of
one. If that file changes, this training goes stale; re-run the notebook.

In [ ]:
from datasets import Dataset

# Imported from the repo in cell 3 — NOT written here. See the note above.
SYSTEM = prompts.EXTRACTION_SYSTEM

def user_turn(row):
    """The user turn the app really sends: the contract, nothing else.

    `row["instruction"]` is deliberately unused. It is the short instruction
    build_pairs.py stamped on every pair; the live path carries that intent
    inside extraction_user() instead, so using both would train on a prompt
    shape that never reaches the model in production.
    """
    return prompts.extraction_user(row["input"])

def to_text(row):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_turn(row)},
        {"role": "assistant", "content": row["output"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_ds = Dataset.from_list([{"text": to_text(r)} for r in train_rows])
val_ds   = Dataset.from_list([{"text": to_text(r)} for r in val_rows])

lengths = [len(tokenizer(t["text"])["input_ids"]) for t in train_ds]
print(f"{len(train_ds)} training / {len(val_ds)} validation examples")
print(f"token length: min {min(lengths)}, median {sorted(lengths)[len(lengths)//2]}, max {max(lengths)}")
if max(lengths) > MAX_SEQ_LEN:
    print(f"WARNING: {sum(1 for l in lengths if l > MAX_SEQ_LEN)} example(s) will be truncated")

# The exam questions are longer than the training ones. They are not trained on,
# but they run through the same prompt at Phase 11, so check they fit too.
if eval_rows:
    eval_lengths = [len(tokenizer(SYSTEM + user_turn(r))["input_ids"]) for r in eval_rows]
    print(f"exam prompts (not trained on): max {max(eval_lengths)} tokens vs limit {MAX_SEQ_LEN}")

print("\n--- one formatted example ---\n", train_ds[0]["text"][:700])

## 8 · Train

Three passes over 73 examples. **Roughly 20–40 minutes on a T4.**

(Longer than you might expect for 73 examples: each one now carries the app's
full v4 instructions, so an example is ~1,900 tokens rather than ~600. That
cost buys the thing the phase is actually for — a model tuned on the prompt it
will really be asked.)

A checkpoint is written every epoch, so a dropped tab costs one epoch, not the run.

Watch the loss column. It should fall. If it is flat or climbing after the
first epoch, stop and say so — that is a real signal, not something to wait out.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    # `processing_class`, not `tokenizer` — TRL renamed it. Unsloth still
    # rewrites the old name for you, but relying on someone else's shim is one
    # dependency bump away from breaking.
    processing_class = tokenizer,
    train_dataset = train_ds,
    eval_dataset = val_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_length = MAX_SEQ_LEN,            # was max_seq_length; TRL removed that name
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,     # effective batch 8
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 20260817,
        output_dir = "outputs",
        save_strategy = "epoch",             # the dropped-tab insurance
        eval_strategy = "epoch",
        report_to = "none",
    ),
)

stats = trainer.train()
print("\ntraining loss:", stats.training_loss)

## 9 · Does it actually do the job?

One exam question, run through the model you just trained. This is a sanity
check, **not** the measurement — the real comparison is Phase 11, base versus
tuned, over all 22.

In [ ]:
import json, re

FastLanguageModel.for_inference(model)

sample = (eval_rows or val_rows)[0]
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": user_turn(sample)},
]

# Two steps on purpose. apply_chat_template(tokenize=True) hands back a plain
# tensor on transformers 4.x but a dict on 5.x (`return_dict` now defaults to
# True), so `.shape` works on one and blows up on the other. Templating to text
# and tokenising separately behaves identically on both.
text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")
out    = model.generate(**inputs, max_new_tokens=512, do_sample=False)
answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("MODEL SAID:\n", answer[:800])
print("\nEXPECTED:\n", json.dumps(json.loads(sample["output"]), indent=2)[:800])
try:
    json.loads(re.search(r"\{.*\}", answer, re.S).group(0))
    print("\n-> it produced valid JSON")
except Exception:
    print("\n-> NOT valid JSON. Worth a look, but Phase 11 measures this properly.")

## 10 · Save the patch to HuggingFace

Uploads the adapter only (~50–100 MB), not the whole model.

Afterwards, one line changes in FinSight's `.env`:

```
LLM_MODEL=ibrahim404/finsight-qwen2.5-3b
```

In [ ]:
import os

ADAPTER_REPO = "ibrahim404/finsight-qwen2.5-3b"

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass
token = os.environ.get("HF_TOKEN")
assert token, "No HF_TOKEN. Add it to the Secrets panel (key icon) with Notebook access on."

model.save_pretrained("finsight_adapter")
tokenizer.save_pretrained("finsight_adapter")
print("saved locally:", sorted(os.listdir("finsight_adapter")))

model.push_to_hub(ADAPTER_REPO, token=token, private=True)
tokenizer.push_to_hub(ADAPTER_REPO, token=token, private=True)
print(f"\nuploaded -> https://huggingface.co/{ADAPTER_REPO}")

## Done

The patch is on HuggingFace. What happens next, none of which is in this notebook:

1. **Phase 11 measures it** — all 22 exam contracts through base *and* tuned,
   one variable between them.
2. `training/serve_model.py` loads the adapter and serves it under a second name.
3. `LLM_MODEL` changes in FinSight and nothing else does.

**If the result is that fine-tuning did not help, that is a real finding and gets
reported as one** — the project plan says so in as many words. The product runs
on base weights either way.